# Solutions — Delta Lake (Hard 14)

In [ ]:
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable

df_transactions = spark.table("samples.bakehouse.sales_transactions")
base_path = "/tmp/delta_practice"


## Solution 1 — Write and Read a Delta Table

In [ ]:
delta_path = f"{base_path}/transactions"
(
    df_transactions
    .write.format("delta")
    .mode("overwrite")
    .save(delta_path)
)
result_1 = spark.read.format("delta").load(delta_path)


## Solution 2 — Append to a Delta Table

In [ ]:
delta_path = f"{base_path}/transactions"
new_rows = df_transactions.limit(10)
new_rows.write.format("delta").mode("append").save(delta_path)
result_2 = spark.read.format("delta").load(delta_path)


## Solution 3 — Update Rows in a Delta Table

In [ ]:
delta_path = f"{base_path}/transactions"
dt = DeltaTable.forPath(spark, delta_path)
dt.update(
    condition = F.col("paymentMethod") == "CASH",
    set       = {"paymentMethod": F.lit("CASH_UPDATED")}
)
result_3 = spark.read.format("delta").load(delta_path)


## Solution 4 — Delete Rows from a Delta Table

In [ ]:
delta_path = f"{base_path}/transactions"
dt = DeltaTable.forPath(spark, delta_path)
dt.delete(F.col("paymentMethod") == "CASH_UPDATED")
result_4 = spark.read.format("delta").load(delta_path)


## Solution 5 — MERGE (Upsert)

In [ ]:
delta_path  = f"{base_path}/transactions"
dt = DeltaTable.forPath(spark, delta_path)

updates = df_transactions.limit(5).withColumn("product", F.lit("UPSERTED"))

(
    dt.alias("target")
    .merge(updates.alias("source"), "target.transactionID = source.transactionID")
    .whenMatchedUpdate(set={"product": "source.product"})
    .whenNotMatchedInsertAll()
    .execute()
)
result_5 = spark.read.format("delta").load(delta_path)


## Solution 6 — Time Travel — Read an Earlier Version

In [ ]:
delta_path = f"{base_path}/transactions"
result_6 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)


## Solution 7 — DESCRIBE HISTORY

In [ ]:
delta_path = f"{base_path}/transactions"
dt = DeltaTable.forPath(spark, delta_path)
result_7 = (
    dt.history()
    .select("version", "timestamp", "operation")
    .orderBy(F.col("version").desc())
)


## Solution 8 — Schema Enforcement and Evolution

In [ ]:
delta_path = f"{base_path}/transactions"
# Add a new column (requires mergeSchema option)
with_discount = df_transactions.withColumn("discount", F.lit(0.0))
(
    with_discount.write.format("delta")
    .option("mergeSchema", "true")
    .mode("append")
    .save(delta_path)
)
result_8 = spark.read.format("delta").load(delta_path).select("transactionID", "product", "discount")
